In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import matplotlib.pyplot as plt

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
torch.manual_seed(42)

In [ ]:
df = pd.read_csv('/kaggle/input/digit-recognizer/train.csv')
print(df.shape)
df.head()

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
fig.suptitle("First 16 Images", fontsize=16)

# Plot the first 16 images from the dataset
for i, ax in enumerate(axes.flat):
    img = df.iloc[i, 1:].values.reshape(28, 28)  # Reshape to 28x28
    ax.imshow(img)  # Display in grayscale
    ax.axis('off')  # Remove axis for a cleaner look
    ax.set_title(f"Label: {df.iloc[i, 0]}")  # Show the label

plt.tight_layout(rect=[0, 0, 1, 0.96])  # Adjust layout to fit the title
plt.show()

In [ ]:
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values
print(X.shape)
print(y.shape)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size = 0.2, random_state=42)

In [ ]:
X_train = X_train/255.0
X_val = X_val/255.0

In [ ]:
X_train

In [ ]:
class myDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32).reshape(-1,1,28,28) # -1 automatically infers batchsize, [batch_size,channels,height,width]
        # This makes the data compatible with nn.Conv2d, which expects 4D input.
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [ ]:
train_dataset = myDataset(X_train, y_train)
val_dataset = myDataset(X_val, y_val)

In [ ]:
len(val_dataset)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)

In [ ]:
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [ ]:
class NN(nn.Module):
    def __init__(self, input_features):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(input_features, 32, kernel_size=3, padding='same'), 
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(kernel_size=2, stride=2), 

            nn.Conv2d(32, 64, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(7*7*64, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),

            nn.Linear(64, 10),
            
        )# If you're using nn.CrossEntropyLoss, it expects raw logits of shape [batch_size, num_classes], so you don’t need to apply Softmax manually.

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
learning_rate = 0.001
epochs = 100

In [ ]:
model = NN(1) # cause the image is grayscale, if it was RGB, in_features=3
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)

In [ ]:
for epoch in range(epochs):
    total_epoch_loss = 0
    for batch_features, batch_labels in train_loader:
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        outputs = model(batch_features)
        loss = criterion(outputs, batch_labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_epoch_loss += loss

    avg_loss = total_epoch_loss/len(train_loader)
    print(f'Epoch: {epoch}, loss: {avg_loss}')

In [ ]:
model.eval()

In [ ]:
batch_labels.shape

In [ ]:
total = 0
correct = 0

with torch.no_grad():
    for batch_features, batch_labels in train_loader:
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        outputs = model(batch_features)
        _, predicted = torch.max(outputs, 1) # max_values, max_indices = torch.max(input_tensor, dim=1) # Max along dimension 1 (columns)
        total += batch_labels.shape[0]
        correct += (predicted == batch_labels).sum().item()

print((correct/total)*100)

In [ ]:
# evaluation on validation data
total = 0
correct = 0

with torch.no_grad():
  for batch_features, batch_labels in val_loader:
    # move data to gpu
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

    outputs = model(batch_features)

    _, predicted = torch.max(outputs, 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

In [ ]:
test_df = pd.read_csv('/kaggle/input/digit-recognizer/test.csv')
test_df.head()

In [ ]:
X_test = (test_df/255.0).values

In [ ]:
X_test

In [ ]:
class testDataset(Dataset):
    def __init__(self, features):
        self.features = torch.tensor(features, dtype=torch.float32).reshape(-1,1,28,28)
        
    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index]

In [ ]:
test_dataset = testDataset(X_test)

In [ ]:
test_loader = DataLoader(test_dataset, batch_size = 32, shuffle=False)

In [ ]:
labels = []
with torch.no_grad():
    for batch_features in test_loader:
        batch_features = batch_features.to(device)
        outputs = model(batch_features)
        _, predicted = torch.max(outputs, 1)
        labels.extend(predicted.cpu().numpy())

In [ ]:
submission_df = pd.read_csv('/kaggle/input/digit-recognizer/sample_submission.csv')
submission_df['Label'] = labels
submission_df.to_csv('submission.csv', index=False)